# NB14 — Multi-TF Deep Dive (Phase C, FX-Modell V1)

**Datum:** 2026-05-27
**Architektur:** Router-kompatibel (ANN-009) — Pipeline ist für `core.router.AssetClass.FX` parametrisiert und in V2 für Crypto/Indices/Commodity wiederverwendbar.

## Mission
Dies ist KEIN PF-Maximierungs-Notebook. Es ist eine **Produktentscheidungs-Plattform**.

Pro TF (5m / 15m / 30m / 1h) bewertet:
1. Klassische Quant-Metriken (PF / WR / MDD / CV / Yearly Stability)
2. **Produkt-Metriken** (Signals/Day, Trade Duration, Alert Frequency, Premium Density, Chart Cleanliness, Session Dependency, Pine UX)
3. Hold-Out-Generalisierung (3 nie trainierte FX-Symbole)
4. SHAP-Stabilität pro TF
5. Pooled vs Single-TF (H4)
6. Cutoff-Variation auf 1h (H5)
7. Quality-Anchor-Check (ANN-010) pro TF
8. **Auto-Decision-Engine** → Default-TF, supported_TFs, User-Profile-Mapping, Alert-Strategie

**Bewertungs-Hierarchie (Nico-Lock):**
Stability > maximaler PF · Konsistenz > Peak-Ergebnisse · Produktqualität > reine Quant-Optimierung

**Hypothesen:** `/research/timeframe_comparisons.md` (H1–H5).

## Section 0 — Setup + Config + Drive Mount

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    # Refresh core code from GitHub
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    # Robust copy: ensure new subdirs exist + copy ALL contents (including dotfiles)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    # Drop stale __pycache__ in Drive core/ (can cause stale imports)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    # Verify new modules are present
    expected_new = [
        f'{PROJECT_ROOT}/core/eval/__init__.py',
        f'{PROJECT_ROOT}/core/eval/tf_pipeline.py',
        f'{PROJECT_ROOT}/core/analysis/product_metrics.py',
        f'{PROJECT_ROOT}/core/analysis/quality_check.py',
    ]
    missing_new = [p for p in expected_new if not Path(p).exists()]
    if missing_new:
        print('SYNC FAILED — missing modules:')
        for p in missing_new:
            print(f'   {p}')
        raise SystemExit('Re-run Section 0 or git pull manually.')
    print('Core code synced from GitHub (new modules verified).')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Module-Cache loeschen UND finder-Cache invalidieren
# (kritisch wenn neue Submodule wie core.eval dazukommen)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

RANDOM_SEED = 42
import numpy as np
np.random.seed(RANDOM_SEED)

RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'],
                                          text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb14_{RUN_TIMESTAMP}_{GIT_COMMIT}'

print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')
print(f'RUN_DATE:      {RUN_DATE}')
print(f'GIT_COMMIT:    {GIT_COMMIT}')
print(f'RANDOM_SEED:   {RANDOM_SEED}')


In [ ]:
# Dependencies
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'shap>=0.45', 'pyarrow>=15.0'],
                   capture_output=True)

import pandas as pd
import lightgbm as lgb
import shap
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import (
    DATA_RAW, DATA_PROCESSED, ARTIFACTS_REPORTS,
    FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES,
    PHASE_C_THRESHOLDS, PRODUCT_METRIC_THRESHOLDS, PINE_BUDGET,
)
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from core.router.asset_detector import AssetClass
from core.eval.tf_pipeline import TFEvalConfig, TFEvalResult, decide_tf_setup
from core.analysis.product_metrics import (
    compute_product_metrics_bundle, evaluate_product_thresholds, pine_ux_score,
)
from core.analysis.quality_check import check_quality_anchor, format_quality_report
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.train.lgbm_trainer import train_lgbm, trading_metrics_from_predictions

print('Imports OK.')
print('  TFEvalConfig:', TFEvalConfig.__name__ if 'TFEvalConfig' in dir() else 'MISSING')


In [ ]:
# Experiment Registry — eine zentrale Quelle
EVAL_CONFIG = TFEvalConfig(
    asset_class      = AssetClass.FX,
    train_symbols    = FX_TRAIN_SYMBOLS,       # EURUSD, USDJPY
    holdout_symbols  = FX_HOLDOUT_SYMBOLS,     # GBPUSD, AUDUSD, USDCHF
    timeframes       = ['5m', '15m', '30m', '1h'],
    feature_cols     = [],                     # populiert in Section 2
    R                = 1.5,
    random_seed      = RANDOM_SEED,
    tier_percentiles = {'standard': 0.90, 'high': 0.97, 'premium': 0.99},
    tier_overrides   = {'1h': {'premium': 0.97}},    # H5: top 3% statt top 1% auf 1h
)
R_VALUE = EVAL_CONFIG.R
WIN_R = R_VALUE
LOSS_R = 1.0

ALL_SYMBOLS = EVAL_CONFIG.train_symbols + EVAL_CONFIG.holdout_symbols

OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb14'
for sub in ('metrics', 'shap', 'summaries', 'config_snapshots', 'product'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Asset class:        {EVAL_CONFIG.asset_class.value}')
print(f'Train symbols:      {EVAL_CONFIG.train_symbols}')
print(f'Hold-Out symbols:   {EVAL_CONFIG.holdout_symbols}')
print(f'Timeframes:         {EVAL_CONFIG.timeframes}')
print(f'Tier overrides:     {EVAL_CONFIG.tier_overrides}')
print(f'Output dir:         {OUTPUT_DIR}')
print(f'Phase-C thresholds: {len(PHASE_C_THRESHOLDS)} keys')
print(f'Product thresholds: {len(PRODUCT_METRIC_THRESHOLDS)} keys')

## Section 1 — Raw OHLCV Inventory + Drive-Restore (NB13-Pattern)

Prüft Roh-OHLCV-Verfügbarkeit. Wir brauchen:
- Primary TFs (`5m, 15m, 30m, 1h`) für jedes Symbol
- HTF-Context TFs (`1h, 4h`) für die HTF-Features

Wenn was fehlt → klarer Fail mit Hinweis welcher Fetcher (NB01) zu starten ist.

In [ ]:
if IN_COLAB:
    DATA_RAW_PATH = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
else:
    DATA_RAW_PATH = DATA_RAW
DATA_RAW_PATH.mkdir(parents=True, exist_ok=True)

needed_tfs_per_symbol = set(EVAL_CONFIG.timeframes) | set(HTF_CONTEXT_TIMEFRAMES)

missing_raw = []
present_raw = []
for s in ALL_SYMBOLS:
    for tf in sorted(needed_tfs_per_symbol):
        p = DATA_RAW_PATH / f'{s}_{tf}.parquet'
        (present_raw if p.exists() else missing_raw).append((s, tf))

print(f'Raw OHLCV inventory:')
print(f'  Needed: {len(ALL_SYMBOLS)} symbols × {len(needed_tfs_per_symbol)} TFs = {len(ALL_SYMBOLS)*len(needed_tfs_per_symbol)} files')
print(f'  Present: {len(present_raw)}')
print(f'  Missing: {len(missing_raw)}')

if missing_raw:
    print(f'\n⚠️  Missing OHLCV files:')
    by_sym = {}
    for s, tf in missing_raw:
        by_sym.setdefault(s, []).append(tf)
    for s, tfs in sorted(by_sym.items()):
        print(f'    {s}: {sorted(tfs)}')
    print(f'\nAction: NB01 (Dukascopy FX fetcher) für fehlende Symbole/TFs starten.')
    raise SystemExit('Raw OHLCV nicht ready. NB01 ausführen und zurückkommen.')
else:
    print('\n✅ Alle Raw-OHLCV-Files vorhanden.')

## Section 2 — Extended Features Build + Drive Restore (NB13-Pattern)

Wir brauchen `{symbol}_{tf}_extended.parquet` mit base + HTF + macro + SMC + session + HTF-interactions + label.
Wenn ein File fehlt: aus Raw OHLCV + Labels bauen. Wenn Labels fehlen: aus Drive-Backup restoren oder NB04 fordern.

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    EXT_DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    
    # Restore labels from Drive backup if local is empty
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        print('Restoring labels from Drive backup...')
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        print('Restoring extended features from Drive backup...')
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = DATA_PROCESSED
    DATA_EXT.mkdir(parents=True, exist_ok=True)

def load_ohlcv_raw(symbol, tf):
    p = DATA_RAW_PATH / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None

def build_extended_features(symbol, tf):
    """Compute extended features for one (symbol, tf) pair, including labels."""
    ohlcv = load_ohlcv_raw(symbol, tf)
    if ohlcv is None or ohlcv.empty:
        return None
    base = compute_features(ohlcv)
    htf_data = {}
    for htf in HTF_CONTEXT_TIMEFRAMES:
        d = load_ohlcv_raw(symbol, htf)
        htf_data[htf] = compute_features(d) if (d is not None and not d.empty) else pd.DataFrame()
    base = attach_htf_context(base, htf_data.get('1h', pd.DataFrame()), htf_data.get('4h', pd.DataFrame()))
    macro_path = DATA_RAW_PATH / 'macro_daily.parquet'
    macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()
    base = attach_macro(base, macro)
    atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
    ema_align = base['ema_alignment'].fillna(0).values if 'ema_alignment' in base.columns else np.zeros(len(base))
    smc = compute_smc_features(ohlcv, atr14, ema_align)
    sess = compute_session_features(ohlcv, atr14)
    inter = compute_htf_interactions(base)
    ext = pd.concat([base, smc, sess, inter], axis=1)
    # Attach labels (mit hit_bar_offset — wird für Produkt-Metriken gebraucht)
    label_path = DATA_PROCESSED_LOCAL / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
    if not label_path.exists():
        return None
    labels = pd.read_parquet(label_path)
    # hit_bar_offset ist optional in alten labels — fallback auf time_barrier_bars (24)
    cols_to_join = ['label']
    if 'hit_bar_offset' in labels.columns:
        cols_to_join.append('hit_bar_offset')
    ext = ext.join(labels[cols_to_join], how='inner')
    if 'hit_bar_offset' not in ext.columns:
        ext['hit_bar_offset'] = 24    # Time-Barrier-Fallback
    ext['symbol'] = symbol
    ext['timeframe'] = tf
    return ext

# Build missing extended files
expected = [(s, tf) for s in ALL_SYMBOLS for tf in EVAL_CONFIG.timeframes]
missing_ext = [(s, tf) for s, tf in expected if not (DATA_EXT / f'{s}_{tf}_extended.parquet').exists()]
print(f'Extended features: {len(expected) - len(missing_ext)} present, {len(missing_ext)} to build')

if missing_ext:
    skipped_labels = []
    for s, tf in tqdm(missing_ext, desc='Building extended'):
        ext = build_extended_features(s, tf)
        if ext is None:
            skipped_labels.append((s, tf))
            continue
        out = DATA_EXT / f'{s}_{tf}_extended.parquet'
        ext.to_parquet(out, compression='zstd')
    if skipped_labels:
        print(f'\n⚠️  {len(skipped_labels)} (symbol, tf) Pairs hatten KEINE Labels:')
        for s, tf in skipped_labels[:10]:
            print(f'    {s} {tf} — labels_{s}_{tf}_R{R_VALUE}.parquet fehlt')
        print(f'\nAction: NB04 (Triple-Barrier-Labeling) für diese Pairs ausführen.')
        raise SystemExit('Labels fehlen — NB04 ausführen, dann NB14 neu starten.')

if IN_COLAB:
    print('Syncing extended features to Drive backup...')
    subprocess.run(['rsync', '-ah', f'{DATA_EXT}/', f'{EXT_DRIVE_BACKUP}/'])

n_now = len(list(DATA_EXT.glob('*_extended.parquet')))
print(f'\n✅ Extended feature files now: {n_now}')

## Section 2b — Feature Column Selection (Phase-1-Winner)

In [ ]:
probe = load_ext(EVAL_CONFIG.train_symbols[0], '5m')
all_cols = [c for c in probe.columns if c not in (NON_FEATURE_COLS | {'hit_bar_offset', 'symbol', 'timeframe'})]

config_path = Path(PROJECT_ROOT) / 'artifacts' / 'reports' / 'phase1_best_config.json'
if config_path.exists():
    with open(config_path) as f:
        feature_cols = json.load(f).get('feature_cols', all_cols)
    feature_cols = [c for c in feature_cols if c in all_cols]
else:
    # NB11-Winner 27 features
    feature_cols = [
        'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
        'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
        'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
        'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
        'htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
        'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
        'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
        'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear',
    ]
    feature_cols = [c for c in feature_cols if c in all_cols]

EVAL_CONFIG.feature_cols = feature_cols
print(f'✅ {len(feature_cols)} features locked for evaluation.')
print(f'Sample: {feature_cols[:6]} ...')

del probe
gc.collect()

## Section 3 — Per-TF Walk-Forward Train + Evaluation Loop

Pro TF: stack train symbols → walk-forward split → train LGBM → VAL-cutoffs → eval auf TEST.

In [ ]:
def stack_pool(symbols, tf):
    frames = []
    for s in symbols:
        d = load_ext(s, tf)
        if d is not None and not d.empty:
            float_cols = d.select_dtypes(include=['float64']).columns
            if len(float_cols) > 0:
                d = d.astype({c: 'float32' for c in float_cols}, copy=False)
            frames.append(d)
    return pd.concat(frames, axis=0).sort_index() if frames else pd.DataFrame()

def compute_yearly_pf(test_df, proba, threshold, R):
    out = {}
    test_df = test_df.copy()
    test_df['_proba'] = proba
    for year, sub in test_df.groupby(test_df.index.year):
        m = sub['_proba'].values >= threshold
        labs = sub['label'].iloc[m.nonzero()[0]]
        wins = int((labs == 1).sum())
        losses = int((labs == -1).sum())
        out[int(year)] = (wins * R) / losses if losses > 0 else (float('inf') if wins > 0 else 0.0)
    return out

def compute_max_drawdown(test_df, proba, threshold, R):
    test_df = test_df.copy()
    test_df['_proba'] = proba
    m = test_df['_proba'].values >= threshold
    labs = test_df['label'].iloc[m.nonzero()[0]]
    pnl = np.where(labs.values == 1, R, np.where(labs.values == -1, -1.0, 0.0))
    if len(pnl) == 0:
        return 0.0
    equity = np.cumsum(pnl)
    peak = np.maximum.accumulate(equity)
    dd = (peak - equity)
    return float(dd.max() / max(peak.max(), 1.0)) if peak.max() > 0 else 0.0

In [ ]:
results_per_tf = {}
models_cache = {}   # für Hold-Out in Section 4 wiederverwenden

for tf in EVAL_CONFIG.timeframes:
    print(f'\n=== TF: {tf} ===')
    pool = stack_pool(EVAL_CONFIG.train_symbols, tf)
    if pool.empty:
        print(f'  ⚠️ no training data — skipping')
        continue
    pool_c = pool.dropna(subset=EVAL_CONFIG.feature_cols + ['label'])
    train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
    del pool, pool_c
    gc.collect()
    print(f'  rows: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

    X_train = train_df[EVAL_CONFIG.feature_cols].values.astype(np.float32)
    y_train_bin = binary_label_for_long(train_df['label']).values
    X_val = val_df[EVAL_CONFIG.feature_cols].values.astype(np.float32)
    y_val_bin = binary_label_for_long(val_df['label']).values

    model = train_lgbm(
        pd.DataFrame(X_train, columns=EVAL_CONFIG.feature_cols), pd.Series(y_train_bin),
        pd.DataFrame(X_val,   columns=EVAL_CONFIG.feature_cols), pd.Series(y_val_bin),
    )
    models_cache[tf] = model     # für Hold-Out reuse

    proba_val = model.predict(X_val)
    X_test = test_df[EVAL_CONFIG.feature_cols].values.astype(np.float32)
    proba_test = model.predict(X_test)

    cutoffs = {
        tier: float(np.quantile(proba_val, EVAL_CONFIG.cutoff_for(tf, tier)))
        for tier in ('standard', 'high', 'premium')
    }
    print(f'  cutoffs: standard={cutoffs["standard"]:.4f}  high={cutoffs["high"]:.4f}  premium={cutoffs["premium"]:.4f}')

    res = TFEvalResult(
        asset_class       = EVAL_CONFIG.asset_class.value,
        tf                = tf,
        n_train_symbols   = len(EVAL_CONFIG.train_symbols),
        n_holdout_symbols = len(EVAL_CONFIG.holdout_symbols),
        n_train_rows      = len(train_df),
        n_val_rows        = len(val_df),
        n_test_rows       = len(test_df),
        cutoffs           = cutoffs,
    )

    for tier in ('standard', 'high', 'premium'):
        m = trading_metrics_from_predictions(test_df['label'], proba_test, cutoffs[tier], R_VALUE)
        m['max_drawdown'] = compute_max_drawdown(test_df, proba_test, cutoffs[tier], R_VALUE)
        setattr(res, f'quant_{tier}', m)

    res.yearly_pf = compute_yearly_pf(test_df, proba_test, cutoffs['premium'], R_VALUE)
    pfs = [v for v in res.yearly_pf.values() if np.isfinite(v) and v > 0]
    res.stability_cv = float(np.std(pfs) / np.mean(pfs)) if pfs and np.mean(pfs) > 0 else 0.0

    print(f'  Premium PF: {res.quant_premium["profit_factor"]:.3f}  '
          f'WR: {res.quant_premium["win_rate"]:.2%}  '
          f'n: {res.quant_premium["n_trades"]:,}  '
          f'CV: {res.stability_cv:.3f}  '
          f'MDD: {res.quant_premium["max_drawdown"]:.1%}')

    # SHAP
    sample_n = min(5000, len(X_test))
    sample_idx = np.random.choice(len(X_test), sample_n, replace=False)
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_test[sample_idx])
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1] if len(shap_vals) > 1 else shap_vals[0]
    mean_abs = np.abs(shap_vals).mean(axis=0)
    shap_pairs = sorted(zip(EVAL_CONFIG.feature_cols, mean_abs), key=lambda x: -x[1])
    res.shap_top = shap_pairs[:10]
    print(f'  SHAP Top-3: {[(n, round(float(v), 4)) for n, v in shap_pairs[:3]]}')

    # Produkt-Metriken (Premium + High Tier)
    timestamps_test = test_df.index
    hit_offsets = test_df['hit_bar_offset'].values
    res.product_premium = compute_product_metrics_bundle(
        proba_test, cutoffs['premium'], timestamps_test, hit_offsets,
        tf=tf, n_symbols=len(EVAL_CONFIG.train_symbols),
    )
    res.product_high = compute_product_metrics_bundle(
        proba_test, cutoffs['high'], timestamps_test, hit_offsets,
        tf=tf, n_symbols=len(EVAL_CONFIG.train_symbols),
    )
    res.product_thresholds_check = evaluate_product_thresholds(
        res.product_premium, PRODUCT_METRIC_THRESHOLDS, tier='premium',
    )
    print(f'  Premium sigs/day/sym: {res.product_premium["signals_per_day_per_symbol"]:.2f}  '
          f'session-dep: {res.product_premium["session_dependency"]:.2f}  '
          f'verdict: {res.product_thresholds_check["verdict"]}')

    # Pine UX
    res.pine_ux = pine_ux_score(
        tf, n_features=len(EVAL_CONFIG.feature_cols),
        n_trees=30, tree_depth=3, requests_security_count=2,
        pine_budget=PINE_BUDGET,
    )
    print(f'  Pine ops/bar est: {res.pine_ux["ops_per_bar_estimate"]}  '
          f'budget util: {res.pine_ux["budget_utilization"]:.2%}')

    # Quality Anchor (Holdout-Werte werden in Section 4 nachgereicht)
    qa_input = {
        'premium_pf_mean_oos':         res.quant_premium['profit_factor'],
        'premium_pf_holdout_mean':     0,
        'min_pf_per_symbol':           res.quant_premium['profit_factor'],
        'stability_cv':                res.stability_cv,
        'min_pf_per_year':             min(pfs) if pfs else 0,
        'min_trades_per_year_tier':    res.quant_premium['n_trades'] / max(1, len(res.yearly_pf)),
        'premium_wr':                  res.quant_premium['win_rate'],
    }
    _, _, res.quality_anchor = check_quality_anchor(qa_input, asset_class='fx', pine_budget_ok=True)

    results_per_tf[tf] = res
    del train_df, val_df, test_df, X_train, X_val, X_test
    del proba_val, proba_test, shap_vals
    gc.collect()

print(f'\n✅ {len(results_per_tf)} TFs evaluated.')

## Section 4 — Per-TF Hold-Out Test (GBPUSD, AUDUSD, USDCHF)

Wiederverwendung der trainierten Modelle aus `models_cache`.

In [ ]:
for tf, res in results_per_tf.items():
    print(f'\n=== Hold-Out TF: {tf} ===')
    model = models_cache.get(tf)
    if model is None:
        print('  no cached model, skip')
        continue
    premium_cutoff = res.cutoffs['premium']
    per_sym_pfs = []
    for sym in EVAL_CONFIG.holdout_symbols:
        h = load_ext(sym, tf)
        if h is None or h.empty:
            print(f'  {sym}: kein Extended-File')
            continue
        h = h.dropna(subset=EVAL_CONFIG.feature_cols + ['label'])
        _, _, h_test = walk_forward_split(h, TRAIN_END, VAL_END)
        if h_test.empty:
            continue
        X_h = h_test[EVAL_CONFIG.feature_cols].values.astype(np.float32)
        p_h = model.predict(X_h)
        m = trading_metrics_from_predictions(h_test['label'], p_h, premium_cutoff, R_VALUE)
        res.holdout_per_symbol[sym] = m
        pf_val = m['profit_factor'] if np.isfinite(m['profit_factor']) else 0
        per_sym_pfs.append(pf_val)
        print(f'  {sym}: PF={m["profit_factor"]:.3f}  WR={m["win_rate"]:.2%}  n={m["n_trades"]}')
        del h, h_test, X_h, p_h
        gc.collect()
    if per_sym_pfs:
        res.holdout_premium = {
            'profit_factor':       float(np.mean(per_sym_pfs)),
            'profit_factor_min':   float(np.min(per_sym_pfs)),
            'profit_factor_max':   float(np.max(per_sym_pfs)),
        }
        # Quality-Anchor refresh
        qa_input = dict(res.quality_anchor.get('metrics_input', {}))
        qa_input['premium_pf_holdout_mean'] = res.holdout_premium['profit_factor']
        qa_input['min_pf_per_symbol']       = float(np.min(per_sym_pfs))
        _, _, res.quality_anchor = check_quality_anchor(qa_input, asset_class='fx', pine_budget_ok=True)
        print(f'  HOLD-OUT MEAN PF: {res.holdout_premium["profit_factor"]:.3f}  '
              f'min: {res.holdout_premium["profit_factor_min"]:.3f}')

## Section 5 — Summary Table (Quant + Produkt + Quality)

In [ ]:
rows = []
for tf, r in results_per_tf.items():
    rows.append({
        'tf': tf,
        'premium_pf':            r.quant_premium.get('profit_factor', 0),
        'premium_wr':            r.quant_premium.get('win_rate', 0),
        'premium_n_trades':      r.quant_premium.get('n_trades', 0),
        'premium_mdd':           r.quant_premium.get('max_drawdown', 0),
        'stability_cv':          r.stability_cv,
        'holdout_pf':            r.holdout_premium.get('profit_factor', 0),
        'holdout_pf_min':        r.holdout_premium.get('profit_factor_min', 0),
        'sigs_day_premium':      r.product_premium.get('signals_per_day_per_symbol', 0),
        'session_dep':           r.product_premium.get('session_dependency', 0),
        'pine_budget_util':      r.pine_ux.get('budget_utilization', 0),
        'product_verdict':       r.product_thresholds_check.get('verdict', '?'),
        'quality_severity':      r.quality_anchor.get('severity', '?'),
    })
summary = pd.DataFrame(rows).round(4)
summary

## Section 6 — SHAP per TF

In [ ]:
shap_rows = []
for tf, r in results_per_tf.items():
    for rank, (name, val) in enumerate(r.shap_top[:5], 1):
        shap_rows.append({'tf': tf, 'rank': rank, 'feature': name, 'mean_abs_shap': float(val)})
shap_df = pd.DataFrame(shap_rows)
if not shap_df.empty:
    shap_pivot = shap_df.pivot(index='tf', columns='rank', values='feature')
    shap_pivot.columns = [f'top_{c}' for c in shap_pivot.columns]
    display(shap_pivot)
shap_df.head(20)

## Section 7 — Pooled vs Single-TF Test (H4)

In [ ]:
pooled_frames = []
for tf in EVAL_CONFIG.timeframes:
    p = stack_pool(EVAL_CONFIG.train_symbols, tf)
    if p.empty:
        continue
    p['_tf'] = tf
    pooled_frames.append(p)

if pooled_frames:
    pooled = pd.concat(pooled_frames, axis=0).sort_index()
    pooled = pooled.dropna(subset=EVAL_CONFIG.feature_cols + ['label'])
    tr_p, va_p, te_p = walk_forward_split(pooled, TRAIN_END, VAL_END)
    del pooled, pooled_frames
    gc.collect()
    print(f'Pooled: train={len(tr_p):,}  val={len(va_p):,}  test={len(te_p):,}')

    X_tp = tr_p[EVAL_CONFIG.feature_cols].values.astype(np.float32)
    y_tp = binary_label_for_long(tr_p['label']).values
    X_vp = va_p[EVAL_CONFIG.feature_cols].values.astype(np.float32)
    y_vp = binary_label_for_long(va_p['label']).values
    pooled_model = train_lgbm(
        pd.DataFrame(X_tp, columns=EVAL_CONFIG.feature_cols), pd.Series(y_tp),
        pd.DataFrame(X_vp, columns=EVAL_CONFIG.feature_cols), pd.Series(y_vp),
    )
    proba_vp = pooled_model.predict(X_vp)
    pooled_premium_cutoff = float(np.quantile(proba_vp, EVAL_CONFIG.tier_percentiles['premium']))

    h4_rows = []
    for tf in EVAL_CONFIG.timeframes:
        sub = te_p[te_p['_tf'] == tf]
        if sub.empty:
            continue
        X_sub = sub[EVAL_CONFIG.feature_cols].values.astype(np.float32)
        p_sub = pooled_model.predict(X_sub)
        m_pool = trading_metrics_from_predictions(sub['label'], p_sub, pooled_premium_cutoff, R_VALUE)
        m_single = results_per_tf[tf].quant_premium
        h4_rows.append({
            'tf': tf,
            'single_tf_pf': m_single.get('profit_factor', 0),
            'pooled_pf':    m_pool.get('profit_factor', 0),
            'lift':         m_pool.get('profit_factor', 0) - m_single.get('profit_factor', 0),
            'single_n':     m_single.get('n_trades', 0),
            'pooled_n':     m_pool.get('n_trades', 0),
        })
    h4_df = pd.DataFrame(h4_rows).round(4)
    del pooled_model, X_tp, X_vp, te_p, tr_p, va_p, proba_vp
    gc.collect()
else:
    h4_df = pd.DataFrame()
h4_df

## Section 8 — Cutoff-Variation auf 1h (H5)

In [ ]:
h5_pass = False
if '1h' in results_per_tf:
    r1h = results_per_tf['1h']
    n_trades_year = r1h.quant_premium.get('n_trades', 0) / max(1, len(r1h.yearly_pf))
    h5_pass = (
        r1h.quant_premium.get('profit_factor', 0) >= 1.5
        and n_trades_year >= PHASE_C_THRESHOLDS['h5_min_trades_per_year']
    )
    print(f'1h top-3%-cutoff applied (tier_override). '
          f'PF={r1h.quant_premium["profit_factor"]:.3f}  '
          f'trades/year≈{n_trades_year:.0f}  H5 PASS={h5_pass}')
else:
    print('1h nicht im Run — H5 nicht evaluierbar')

## Section 9 — Auto-Decision-Engine (PRODUKT-ENTSCHEIDUNGEN)

In [ ]:
decision = decide_tf_setup(results_per_tf, PHASE_C_THRESHOLDS)

print('=' * 70)
print('PRODUKT-ENTSCHEIDUNGEN — NB14')
print('=' * 70)
print(f'\nDefault-TF (Balanced):         {decision["default_tf"]}')
print(f'Supported TFs in V1:           {decision["supported_tfs"]}')
print(f'\nUser-Profile-Mapping:')
for prof, tf in decision['profile_map'].items():
    sigs = results_per_tf[tf].product_premium.get('signals_per_day_per_symbol', 0) if tf in results_per_tf else 0
    print(f'  {prof:13s} → {tf:4s}  ({sigs:.1f} Premium-Signale/Tag/Symbol)')
print(f'\nAlert-Strategie:')
for tf, strat in decision['alert_strategy'].items():
    print(f'  {tf:4s}: alert on "{strat["alert_on_tier"]}" tier (~{strat["expected_alerts_per_day"]:.1f}/day/sym)')
print(f'\nHypothesen:')
print(f'  H1 (5m=Default):                {"PASS" if decision["h1_pass"] else "FAIL"}')
print(f'  H2 (15m=Conservative):          {"PASS" if decision["h2_pass"] else "FAIL"}')
print(f'  H3 (30m/1h aus V1):             {"PASS" if decision["h3_pass"] else "FAIL"}')
print(f'  H4 (Pooled > Single-TF):        siehe Section 7')
print(f'  H5 (1h Top-3% revive):          {"PASS" if h5_pass else "FAIL"}')
print(f'\nBegründung:')
for line in decision['explanation']:
    print(f'  {line}')

## Section 10 — Quality Anchor Reports (ANN-010)

In [ ]:
for tf, r in results_per_tf.items():
    print(f'\n--- {tf} ---')
    print(format_quality_report(r.quality_anchor))

## Section 11 — Result Persistence

In [ ]:
summary.to_csv(OUTPUT_DIR / 'metrics' / f'per_tf_summary_{RUN_DATE}.csv', index=False)
if not shap_df.empty:
    shap_df.to_csv(OUTPUT_DIR / 'shap' / f'shap_per_tf_{RUN_DATE}.csv', index=False)
if not h4_df.empty:
    h4_df.to_csv(OUTPUT_DIR / 'summaries' / f'pooled_vs_single_tf_{RUN_DATE}.csv', index=False)

snapshot = {
    'experiment_id':    EXPERIMENT_ID,
    'run_date':         RUN_DATE,
    'git_commit':       GIT_COMMIT,
    'asset_class':      EVAL_CONFIG.asset_class.value,
    'train_symbols':    EVAL_CONFIG.train_symbols,
    'holdout_symbols':  EVAL_CONFIG.holdout_symbols,
    'timeframes':       EVAL_CONFIG.timeframes,
    'feature_cols':     EVAL_CONFIG.feature_cols,
    'tier_percentiles': EVAL_CONFIG.tier_percentiles,
    'tier_overrides':   EVAL_CONFIG.tier_overrides,
    'phase_c_thresholds': PHASE_C_THRESHOLDS,
    'product_thresholds': PRODUCT_METRIC_THRESHOLDS,
    'results':          {tf: r.to_dict() for tf, r in results_per_tf.items()},
    'pooled_vs_single': h4_df.to_dict(orient='records') if not h4_df.empty else [],
    'h5_pass':          h5_pass,
    'decision':         decision,
}
with open(OUTPUT_DIR / 'summaries' / f'nb14_full_snapshot_{RUN_DATE}.json', 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)

with open(OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json', 'w') as f:
    json.dump({
        'experiment_id':    EXPERIMENT_ID,
        'run_date':         RUN_DATE,
        'git_commit':       GIT_COMMIT,
        'asset_class':      EVAL_CONFIG.asset_class.value,
        'train_symbols':    EVAL_CONFIG.train_symbols,
        'holdout_symbols':  EVAL_CONFIG.holdout_symbols,
        'timeframes':       EVAL_CONFIG.timeframes,
        'feature_count':    len(EVAL_CONFIG.feature_cols),
        'tier_percentiles': EVAL_CONFIG.tier_percentiles,
        'tier_overrides':   EVAL_CONFIG.tier_overrides,
    }, f, indent=2)

print(f'✅ Results written to {OUTPUT_DIR}')
print(f'   per_tf_summary_{RUN_DATE}.csv')
print(f'   shap_per_tf_{RUN_DATE}.csv')
print(f'   pooled_vs_single_tf_{RUN_DATE}.csv')
print(f'   nb14_full_snapshot_{RUN_DATE}.json')
print(f'   {EXPERIMENT_ID}_config.json')

## Section 12 — Auto-Push to GitHub

Setup: `docs/colab_auto_push.md` — Token `GITHUB_TOKEN` in Colab Secrets.

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip auto-push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        try:
            from google.colab import userdata
            GH_TOKEN = userdata.get('GH_PAT')
        except Exception:
            GH_TOKEN = None

    if not GH_TOKEN:
        print('⚠️ GITHUB_TOKEN nicht gefunden — Results bleiben im Drive.')
    else:
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)
        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file():
                continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        print(f'Copied {len(copied)} files')
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--quiet', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        st = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'], capture_output=True, text=True)
        if not st.stdout.strip():
            print('Nothing to commit.')
        else:
            msg = (f'NB14 results: Multi-TF deep dive {RUN_DATE} ({len(copied)} files)\n\n'
                   f'Default-TF: {decision["default_tf"]}  '
                   f'Supported: {decision["supported_tfs"]}  '
                   f'H1={decision["h1_pass"]} H2={decision["h2_pass"]} H3={decision["h3_pass"]}')
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                  capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'✓ Pushed as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)

## Section 13 — Final Verdict

Diese Outputs aus Sections 5 + 9 + 10 sind die Eingaben für die nächsten Doku-Updates (Claude macht das nach dem Run):

- `docs/decisions/ANN-011-XX.md` — finales V1-TF-Setup als ADR
- `docs/roadmap.md` — Phase D Start mit klarem TF-Mandat
- `docs/model_registry.md` — FX-Modell TF-Setup gelocked
- `docs/pine_router_design.md` — TF-Handling in Pine-Code
- `research/timeframe_comparisons.md` — Section 3 (Resultat) + 4 (Decision) + 5 (Konsequenz)